# Import libraries and initialize Groq

In [1]:
import os
import json
import time
import pandas as pd
import numpy as np

from datetime import datetime, timezone
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

MODEL_NAME = "openai/gpt-oss-20b"

print("Groq client initialized successfully!")

Groq client initialized successfully!


# Load Questions

In [2]:
with open("../data/questions.json", "r") as f:
    questions = json.load(f)

print(f"Total questions loaded: {len(questions)}")

for q in questions:
    print(q["id"], ":", q["question"])

Total questions loaded: 5
Q1 : What is machine learning?
Q2 : Explain the difference between supervised and unsupervised learning.
Q3 : What is overfitting in machine learning and how can it be prevented?
Q4 : Explain how a neural network works.
Q5 : What is the difference between machine learning and deep learning?


# Groq API calling Function

In [5]:
def call_llm(question, prompt_version="v1"):

    prompt = f"""
    Answer the following student question clearly.

    Question: {question}
    """

    start_time = datetime.now(timezone.utc)
    start = time.perf_counter()

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )

        end = time.perf_counter()
        end_time = datetime.now(timezone.utc)

        return {
            "response": response.choices[0].message.content,
            "latency": end - start,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "question": question,
            "prompt_version": prompt_version,
            "model_name": MODEL_NAME,
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
            "response_status": "success"
        }

    except Exception as e:
        end = time.perf_counter()
        end_time = datetime.now(timezone.utc)

        return {
            "response": None,
            "latency": end - start,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "question": question,
            "prompt_version": prompt_version,
            "model_name": MODEL_NAME,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "response_status": f"failed: {str(e)}"
        }

# testing the API

In [6]:
result = call_llm(
    "What is machine learning?"
)

print("Response:", result["response"])
print("Latency:", round(result["latency"], 3), "seconds")
print("Input tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])
print("Total tokens:", result["total_tokens"])

Response: **Machine learning** is a branch of computer science that teaches computers to learn patterns and make decisions from data, rather than being explicitly programmed for every task.

Key points:

1. **Learning from data** – A machine learning model is trained on a dataset (examples with known outcomes). It identifies regularities or relationships in that data.

2. **Making predictions or decisions** – After training, the model can apply what it learned to new, unseen data to predict outcomes, classify items, or recommend actions.

3. **Types of learning**  
   - **Supervised learning**: The data includes correct answers (labels). The model learns to map inputs to outputs (e.g., spam detection).  
   - **Unsupervised learning**: No labels are provided. The model discovers hidden structure, such as grouping similar customers together.  
   - **Reinforcement learning**: The model learns by trial and error, receiving rewards or penalties for actions (e.g., game playing, robotics).


# Execute the five academic questions

In [7]:
results = []

for item in questions:

    print(f"Processing {item['id']}...")

    result = call_llm(
        item["question"],
        prompt_version="v1"
    )

    result["question_id"] = item["id"]

    results.append(result)

    print("Status:", result["response_status"])
    print("Latency:", round(result["latency"], 3), "seconds")
    print("-" * 50)

Processing Q1...
Status: success
Latency: 0.584 seconds
--------------------------------------------------
Processing Q2...
Status: success
Latency: 0.944 seconds
--------------------------------------------------
Processing Q3...
Status: success
Latency: 1.303 seconds
--------------------------------------------------
Processing Q4...
Status: success
Latency: 1.446 seconds
--------------------------------------------------
Processing Q5...
Status: success
Latency: 0.935 seconds
--------------------------------------------------


# Latency statistics

In [8]:
df = pd.DataFrame(results)

latency_df = df[
    [
        "question_id",
        "prompt_version",
        "model_name",
        "start_time",
        "end_time",
        "latency",
        "response_status"
    ]
]

display(latency_df)

,question_id,prompt_version,model_name,start_time,end_time,latency,response_status
0,Q1,v1,openai/gpt-oss-20b,2026-10-31T07:00:35.526450+00:00,2026-10-31T07:00:36.110043+00:00,0.583572,success
1,Q2,v1,openai/gpt-oss-20b,2026-10-31T07:00:36.110695+00:00,2026-10-31T07:00:37.055150+00:00,0.944444,success
2,Q3,v1,openai/gpt-oss-20b,2026-10-31T07:00:37.056096+00:00,2026-10-31T07:00:38.359108+00:00,1.303002,success
3,Q4,v1,openai/gpt-oss-20b,2026-10-31T07:00:38.359671+00:00,2026-10-31T07:00:39.805460+00:00,1.445778,success
4,Q5,v1,openai/gpt-oss-20b,2026-10-31T07:00:39.806790+00:00,2026-10-31T07:00:40.742203+00:00,0.935400,success


In [9]:
latency_stats = {
    "Mean": df["latency"].mean(),
    "Median": df["latency"].median(),
    "Minimum": df["latency"].min(),
    "Maximum": df["latency"].max(),
    "Standard Deviation": df["latency"].std()
}

latency_summary = pd.DataFrame(
    latency_stats.items(),
    columns=["Metric", "Latency (seconds)"]
)

display(latency_summary.round(4))

,Metric,Latency (seconds)
0,Mean,1.0424
1,Median,0.9444
2,Minimum,0.5836
3,Maximum,1.4458
4,Standard Deviation,0.3399


# Price Definition

In [10]:
INPUT_PRICE_PER_MILLION = 0.075
OUTPUT_PRICE_PER_MILLION = 0.30

## Request Level Cost

The formulas are:

**Input Cost:**

$$
InputCost = \frac{InputTokens}{1,000,000}
\times InputPrice
$$

**Output Cost:**

$$
OutputCost = \frac{OutputTokens}{1,000,000}
\times OutputPrice
$$

**Total Cost per Request:**

$$
TotalCost = InputCost + OutputCost
$$



In [11]:
df["input_cost_usd"] = (
    df["input_tokens"] / 1_000_000
) * INPUT_PRICE_PER_MILLION

df["output_cost_usd"] = (
    df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE_PER_MILLION

df["total_cost_usd"] = (
    df["input_cost_usd"] + df["output_cost_usd"]
)

## Projected Monthly Cost

In [12]:
successful_requests = df[
    df["response_status"] == "success"
]

average_cost = successful_requests[
    "total_cost_usd"
].mean()

monthly_requests = 10_000

monthly_projected_cost = (
    average_cost * monthly_requests
)

print("Average cost per request: $", average_cost)
print("Projected monthly cost: $", monthly_projected_cost)

Average cost per request: $ 0.00018175499999999997
Projected monthly cost: $ 1.8175499999999998


# Generate CSV

In [13]:
os.makedirs("../outputs", exist_ok=True)

df.to_csv(
    "../outputs/latency_tokens_cost.csv",
    index=False
)

print("CSV exported successfully!")

CSV exported successfully!


In [14]:
slowest = successful_requests.loc[
    successful_requests["latency"].idxmax()
]

print("Slowest Question:", slowest["question_id"])
print("Question:", slowest["question"])
print("Latency:", round(slowest["latency"], 4), "seconds")
print("Output Tokens:", slowest["output_tokens"])

Slowest Question: Q4
Question: Explain how a neural network works.
Latency: 1.4458 seconds
Output Tokens: 1027


In [15]:
latency_summary.to_csv(
    "../outputs/latency_summary.csv",
    index=False
)